# Parte 3 — Novas variáveis

Cria `df_enriquecido` com as variáveis derivadas do exercício e valida o alvo.


## 3.0 Base tratada (refaz a Parte 2)

Notebook autônomo: reconstrói `df_limpo` para poder rodar isolado.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
def ler_csv_prf(caminho: Path) -> pd.DataFrame:
    """Lê o CSV tentando vários encodings; só falha se todos falharem."""
    for encoding in ("utf-8", "latin1", "utf-8-sig", "cp1252"):
        try:
            df = pd.read_csv(caminho, sep=SEPARADOR, encoding=encoding,
                             low_memory=False, dtype={"id": "string"})
        except (UnicodeDecodeError, UnicodeError):
            continue
        print(f"{caminho.name}: encoding={encoding}, {len(df):,} linhas")
        return df
    raise RuntimeError(f"Nenhum encoding funcionou para {caminho}")


def normalizar_nome(nome: str) -> str:
    nome = str(nome).strip().lower()
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return nome.replace(" ", "_").replace("-", "_").replace("/", "_").strip("_")


In [3]:
dados_por_ano = {}
for ano in ANOS:
    df = ler_csv_prf(DIR_BRUTOS / f"datatran{ano}.csv")
    df.columns = [normalizar_nome(c) for c in df.columns]
    df["arquivo_origem"] = ano
    dados_por_ano[ano] = df

df_limpo = pd.concat([dados_por_ano[a] for a in ANOS], ignore_index=True)
df_limpo = df_limpo.drop_duplicates()

NUMERICAS = ["br", "km", "pessoas", "mortos", "feridos", "feridos_leves",
             "feridos_graves", "ilesos", "ignorados", "veiculos"]
for c in NUMERICAS:
    df_limpo[c] = pd.to_numeric(
        df_limpo[c].astype("string").str.replace(",", ".", regex=False), errors="coerce"
    )
df_limpo["data_inversa"] = pd.to_datetime(
    df_limpo["data_inversa"], format="%Y-%m-%d", errors="coerce"
)

for c in df_limpo.columns:
    if str(df_limpo[c].dtype) in ("object", "string", "str"):
        df_limpo[c] = (df_limpo[c].astype("string").str.strip()
                       .str.replace(r"\s{2,}", " ", regex=True))

# strip + upper nos identificadores (já vêm em caixa alta da fonte); as
# descritivas mantêm a caixa publicada pela PRF.
for c in ("uf", "br", "municipio"):
    if c in df_limpo.columns and str(df_limpo[c].dtype) in ("object", "string", "str"):
        df_limpo[c] = df_limpo[c].astype("string").str.upper()

DOMINIOS = {
    "uf": {"AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS",
           "MT","PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC",
           "SE","SP","TO"},
    "fase_dia": {"Amanhecer", "Anoitecer", "Plena Noite", "Pleno dia"},
    "tipo_pista": {"Dupla", "Múltipla", "Simples"},
    "uso_solo": {"Não", "Sim"},
    "sentido_via": {"Crescente", "Decrescente", "Não Informado"},
    "classificacao_acidente": {"Com Vítimas Fatais", "Com Vítimas Feridas", "Sem Vítimas"},
    "dia_semana": {"domingo", "segunda-feira", "terça-feira", "quarta-feira",
                   "quinta-feira", "sexta-feira", "sábado"},
}
for coluna, dominio in DOMINIOS.items():
    invalidas = df_limpo[coluna].isna() | ~df_limpo[coluna].isin(dominio)
    df_limpo.loc[invalidas, coluna] = "IGNORADO"

df_limpo = df_limpo.drop(
    columns=[c for c in ("latitude", "longitude", "regional", "delegacia", "uop")
             if c in df_limpo.columns]
)
print(df_limpo.shape)


datatran2023.csv: encoding=latin1, 67,766 linhas


datatran2024.csv: encoding=latin1, 73,202 linhas


datatran2025.csv: encoding=latin1, 72,529 linhas


(213497, 26)


## 3.1 `acidente_fatal`

1 se `mortos >= 1`, senão 0.


In [4]:
df_enriquecido = df_limpo.copy()
df_enriquecido["acidente_fatal"] = (df_enriquecido["mortos"] >= 1).astype(int)
print(df_enriquecido["acidente_fatal"].value_counts().to_string())


acidente_fatal
0    198204
1     15293


## 3.2 `ano_acidente` e `mes_acidente`

Extraídos de `data_inversa`. O ano é conferido contra `arquivo_origem`.


In [5]:
df_enriquecido["ano_acidente"] = df_enriquecido["data_inversa"].dt.year
df_enriquecido["mes_acidente"] = df_enriquecido["data_inversa"].dt.month

assert (df_enriquecido["ano_acidente"] == df_enriquecido["arquivo_origem"]).all(), \
    "O ano da data diverge do ano do arquivo"
print(df_enriquecido.groupby(["ano_acidente", "mes_acidente"]).size().unstack().to_string())


mes_acidente    1     2     3     4     5     6     7     8     9     10    11    12
ano_acidente                                                                        
2023          5254  4751  5596  5603  5638  5735  5966  5613  5664  5740  5592  6614
2024          5757  5280  5959  5835  6188  6226  6405  6168  6224  6412  6146  6602
2025          5528  5287  5960  5786  6096  6122  6238  6246  6017  6252  6209  6788


## 3.3 `fim_de_semana`

1 se sábado (5) ou domingo (6), via `dt.dayofweek` (nome atual no pandas 3.x:
`dt.day_of_week`).


In [6]:
df_enriquecido["fim_de_semana"] = (
    df_enriquecido["data_inversa"].dt.day_of_week.isin([5, 6]).astype(int)
)
print(df_enriquecido["fim_de_semana"].value_counts().to_string())

# conferência contra a coluna de texto dia_semana
fds_texto = df_enriquecido["dia_semana"].isin(["sábado", "domingo"]).astype(int)
assert (df_enriquecido["fim_de_semana"] == fds_texto).all()
print("Concorda com dia_semana.")


fim_de_semana
0    144709
1     68788
Concorda com dia_semana.


## 3.4 `data_comemorativa`

| Valor | Regra |
|---|---|
| Fim de Ano | 20/12 a 02/01 |
| Carnaval | 18–22/02/2023, 10–14/02/2024, 01–05/03/2025 |
| Normal | o resto |


In [7]:
CARNAVAL = {
    2023: ("2023-02-18", "2023-02-22"),
    2024: ("2024-02-10", "2024-02-14"),
    2025: ("2025-03-01", "2025-03-05"),
}

mes = df_enriquecido["data_inversa"].dt.month
dia = df_enriquecido["data_inversa"].dt.day
fim_de_ano = ((mes == 12) & (dia >= 20)) | ((mes == 1) & (dia <= 2))

carnaval = pd.Series(False, index=df_enriquecido.index)
for ano, (inicio, fim) in CARNAVAL.items():
    carnaval |= (
        (df_enriquecido["ano_acidente"] == ano)
        & df_enriquecido["data_inversa"].between(inicio, fim)
    )

df_enriquecido["data_comemorativa"] = np.select(
    [fim_de_ano, carnaval], ["Fim de Ano", "Carnaval"], default="Normal"
)
print(df_enriquecido["data_comemorativa"].value_counts().to_string())


data_comemorativa
Normal        201737
Fim de Ano      8863
Carnaval        2897


## 3.5 `faixa_horaria`

Blocos de 3 horas: `00h-03h`, `03h-06h`, … `21h-24h`. O bloco é fechado no início e
aberto no fim — hora 3 cai em `03h-06h`.


In [8]:
hora = pd.to_datetime(
    df_enriquecido["horario"].astype("string").str.strip(),
    format="%H:%M:%S", errors="coerce",
).dt.hour
assert hora.notna().all(), "Há horário inválido"

inicio = (hora // 3) * 3
df_enriquecido["faixa_horaria"] = (
    inicio.map(lambda h: f"{h:02d}h-{h + 3:02d}h").astype("string")
)
print(df_enriquecido["faixa_horaria"].value_counts().sort_index().to_string())


faixa_horaria
00h-03h    11741
03h-06h    14792
06h-09h    33733
09h-12h    27142
12h-15h    28421
15h-18h    36959
18h-21h    39758
21h-24h    20951


## 3.6 Validação do alvo (`assert`)

Zero violações da regra `acidente_fatal == (mortos >= 1)`.


In [9]:
violacoes = int(
    (df_enriquecido["acidente_fatal"] != (df_enriquecido["mortos"] >= 1).astype(int)).sum()
)
assert violacoes == 0, f"{violacoes} linhas violam a regra do alvo"

print(f"Linhas          : {len(df_enriquecido):,}")
print(f"Acidentes fatais: {int(df_enriquecido['acidente_fatal'].sum()):,}")
print(f"Taxa de fatais  : {df_enriquecido['acidente_fatal'].mean():.2%}")
print(f"Colunas         : {df_enriquecido.shape[1]}")
print("\nZero violações da regra do alvo.")


Linhas          : 213,497
Acidentes fatais: 15,293
Taxa de fatais  : 7.16%
Colunas         : 32

Zero violações da regra do alvo.
